In [1]:
import pandas as pd
import sqlite3

1. Create a connection to the database using the sqlite3 library.

In [2]:
db_file = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_file)

2. Get the schema of the test table.

In [3]:
query_schema = "PRAGMA table_info(test);"
schema = pd.read_sql(query_schema, conn)
print("Schema of Test Table:")
print(schema)

Schema of Test Table:
   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


3. Get only the first ten rows of the test table to see what it looks like.

In [4]:
print(pd.read_sql("SELECT * FROM test LIMIT 10", conn))

       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


4. Find the minimum value of the delta between the first commit and the deadline of the corresponding lab for all users using only one query.
- Do this by joining the table with the deadlines table.
- The difference should be displayed in hours.
- Do not take lab project1 into account; it has longer deadlines and will be an outlier.
- The value should be stored in the dataframe df_min with the corresponding uid.

In [7]:
print(pd.read_sql("PRAGMA table_info(deadlines);", conn))

   cid       name     type  notnull dflt_value  pk
0    0      index  INTEGER        0       None   0
1    1       labs     TEXT        0       None   0
2    2  deadlines  INTEGER        0       None   0


In [10]:

query_df_min = """
SELECT t.uid,
MIN((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) AS min_delta_hours
FROM test t
JOIN deadlines d on t.labname = d.labs
WHERE t.labname != 'project1'
GROUP By t.uid;
"""

df_min = pd.read_sql(query_df_min, conn)
df_min.head()

,uid,min_delta_hours
0,user_1,6.796667
1,user_10,39.368056
2,user_14,84.448611
3,user_17,34.643056
4,user_18,3.934167


5. Do the same thing for the maximum, but use only one query. The dataframe name is df_max.

In [13]:
query_df_max = """
SELECT t.uid,
MAX((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) AS max_delta_hours
FROM test t
JOIN deadlines d on t.labname = d.labs
WHERE t.labname != 'project1'
GROUP By t.uid;
"""

df_max = pd.read_sql(query_df_max, conn)
df_max.head()

,uid,max_delta_hours
0,user_1,175.556667
1,user_10,132.341944
2,user_14,200.766389
3,user_17,81.591667
4,user_18,10.973611


6. Do the same thing, but for the average. Use only one query. This time, your dataframe should not include the uid column. The dataframe name is df_avg.

In [15]:
query_df_avg = """
SELECT
AVG((d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0) AS avg_delta_hours
FROM test t
JOIN deadlines d on t.labname = d.labs
WHERE t.labname != 'project1'
GROUP By t.uid;
"""

df_avg = pd.read_sql(query_df_avg, conn)
df_avg.head()

,avg_delta_hours
0,65.119778
1,75.242444
2,159.568796
3,62.207667
4,6.368148


7. We want to test the hypothesis that users who visited the newsfeed just a few times have a lower delta between the first commit and the deadline. To do this, calculate the correlation coefficient between the number of pageviews and the difference.
- Using only one query, create a table with the following columns: "uid", "avg_diff", and "pageviews".
- "uid" is the uids that exist in the test.
- "avg_diff" is the average delta between the first commit and the lab deadline per user.
- "pageviews" is the number of Newsfeed visits per user.
- Do not take the lab project1 into account.
- Store it in the dataframe views_diff.
- Use the Pandas corr() method to calculate the correlation coefficient between the number of pageviews and the difference.

In [20]:
query_views_diff = """
SELECT a.uid, a.avg_diff, b.pageviews
FROM (
SELECT
    t.uid,
    AVG( (d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0 ) AS avg_diff
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
GROUP BY t.uid) a
JOIN (
SELECT
    uid,
    COUNT(*) AS pageviews
FROM pageviews
GROUP BY uid
) b
ON a.uid = b.uid;
"""

views_diff = pd.read_sql(query_views_diff, conn)
views_diff.head()

,uid,avg_diff,pageviews
0,user_1,65.119778,28
1,user_10,75.242444,89
2,user_14,159.568796,143
3,user_17,62.207667,47
4,user_18,6.368148,3


In [21]:
correlation = views_diff['pageviews'].corr(views_diff['avg_diff'])
print("Correlation coefficient:", correlation)

Correlation coefficient: 0.27914346425426556


8. Close the connection.

In [22]:
conn.close()